In [4]:
# =================================================================================================
# NHANES 2015-2016 LAB MERGE - FIDELITY PRESERVING
#
# Source files:
#
#   GLU_I_corrected.csv
#   INS_I_corrected.csv
#   TCHOL_I_corrected.csv
#   FASTQX_I_corrected.csv
#   GHB_I_corrected.csv
#   HDL_I_corrected.csv
#   TRIGLY_I_corrected.csv
#
#
# IMPORTANT 2015-2016 DESIGN CHANGE
# -------------------------------------------------------------------------------------------------
#
# Starting with NHANES 2015-2016:
#
#   GLU_I DOES NOT contain:
#
#       PHAFSTHR
#       PHAFSTMN
#
# CDC states these variables should be obtained from FASTQX_I.
#
#
# INS_I DOES contain:
#
#       WTSAF2YR
#       LBXIN
#       LBDINSI
#       LBDINLC
#       PHAFSTHR
#       PHAFSTMN
#
#
# Therefore:
#
# WTSAF2YR
# ----------
#   Exists in:
#
#       GLU_I
#       INS_I
#       TRIGLY_I
#
#   Strategy:
#
#       compare values by SEQN
#       prove numeric equivalence
#       prove missing-position equivalence
#       keep canonical GLU_I copy
#       remove INS_I / TRIGLY_I redundant copies BEFORE merge
#
#
# PHAFSTHR / PHAFSTMN
# --------------------
#   Exists in:
#
#       INS_I
#       FASTQX_I
#
#   GLU_I does NOT contain them.
#
#   Strategy:
#
#       compare INS_I with FASTQX_I on overlapping participants
#       prove numeric equivalence
#       use FASTQX_I as canonical source because it has broader coverage
#       remove INS_I copies BEFORE merge
#
#
# Fidelity rules:
#
#   - Read corrected CSV files with dtype=str
#   - Preserve all Lab decimal text AS IS
#   - Never round
#   - Never use float_format=
#   - Never integerize genuine Lab measurements
#   - SEQN must remain integer-looking text
#   - Every source must contain unique SEQN
#   - Every participant merge uses validate="one_to_one"
#   - No uncontrolled pandas suffix columns
#   - Validate exact source text for every retained source-specific field
#   - Validate exact pre-save/post-save CSV text
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


LAB_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Labs"
)


GLU_FILE = (
    LAB_DIR
    / "GLU_I_corrected.csv"
)


INS_FILE = (
    LAB_DIR
    / "INS_I_corrected.csv"
)


TCHOL_FILE = (
    LAB_DIR
    / "TCHOL_I_corrected.csv"
)


FASTQX_FILE = (
    LAB_DIR
    / "FASTQX_I_corrected.csv"
)


GHB_FILE = (
    LAB_DIR
    / "GHB_I_corrected.csv"
)


HDL_FILE = (
    LAB_DIR
    / "HDL_I_corrected.csv"
)


TRIGLY_FILE = (
    LAB_DIR
    / "TRIGLY_I_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2015_2016"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_labs_2015_2016_raw_corrected.csv"
)


# =================================================================================================
# 3. VERIFY ALL EXPECTED FILES EXIST
# =================================================================================================

FILES = {

    "GLU_I": GLU_FILE,

    "INS_I": INS_FILE,

    "TCHOL_I": TCHOL_FILE,

    "FASTQX_I": FASTQX_FILE,

    "GHB_I": GHB_FILE,

    "HDL_I": HDL_FILE,

    "TRIGLY_I": TRIGLY_FILE

}


for name, path in FILES.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} corrected file not found:\n{path}"
        )


print(
    "PASS: All seven 2015-2016 Lab source files exist."
)


# =================================================================================================
# 4. READ CORRECTED CSV FILES AS TEXT
# =================================================================================================

#
# CRITICAL:
#
# dtype=str + keep_default_na=False
#
# prevents pandas from re-parsing already-corrected numeric tokens.
#
# Example:
#
#     0.158
#
# remains:
#
#     0.158
#
# and a longer floating representation remains exactly as it
# existed in the corrected source CSV.
#

glucose = pd.read_csv(
    GLU_FILE,
    dtype=str,
    keep_default_na=False
)


insulin = pd.read_csv(
    INS_FILE,
    dtype=str,
    keep_default_na=False
)


chol = pd.read_csv(
    TCHOL_FILE,
    dtype=str,
    keep_default_na=False
)


fasting = pd.read_csv(
    FASTQX_FILE,
    dtype=str,
    keep_default_na=False
)


ghb = pd.read_csv(
    GHB_FILE,
    dtype=str,
    keep_default_na=False
)


hdl = pd.read_csv(
    HDL_FILE,
    dtype=str,
    keep_default_na=False
)


trigly = pd.read_csv(
    TRIGLY_FILE,
    dtype=str,
    keep_default_na=False
)


# =================================================================================================
# 5. DATASET DICTIONARY
# =================================================================================================

datasets = {

    "GLU_I": glucose,

    "INS_I": insulin,

    "TCHOL_I": chol,

    "FASTQX_I": fasting,

    "GHB_I": ghb,

    "HDL_I": hdl,

    "TRIGLY_I": trigly

}


# =================================================================================================
# 6. STANDARDIZE COLUMN NAMES ONLY
# =================================================================================================

#
# Only column-name whitespace/case is changed.
#
# No data value is modified.
#

for name, dataset in datasets.items():

    dataset.columns = (
        dataset.columns
        .str.strip()
        .str.upper()
    )


# =================================================================================================
# 7. DISPLAY SOURCE STRUCTURE
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2015-2016 LAB MERGE"
)


print(
    "=" * 110
)


summary = []


for name, dataset in datasets.items():

    summary.append({

        "file": name,

        "rows": len(dataset),

        "columns": len(dataset.columns),

        "column_names": dataset.columns.tolist()

    })


summary_df = pd.DataFrame(
    summary
)


display(
    summary_df
)


# =================================================================================================
# 8. OFFICIAL CDC SOURCE ROW COUNTS
# =================================================================================================

EXPECTED_SOURCE_ROWS = {

    "GLU_I": 3191,

    "INS_I": 3191,

    "TRIGLY_I": 3191,

    "TCHOL_I": 8021,

    "HDL_I": 8021,

    "GHB_I": 6744,

    "FASTQX_I": 9165

}


print(
    "\n--- OFFICIAL SOURCE ROW COUNTS ---"
)


for name, expected_rows in EXPECTED_SOURCE_ROWS.items():

    actual_rows = len(
        datasets[name]
    )


    print(
        f"{name}: "
        f"expected={expected_rows:,}, "
        f"actual={actual_rows:,}"
    )


    if actual_rows != expected_rows:

        raise ValueError(
            f"{name} row count differs from CDC.\n"
            f"Expected: {expected_rows:,}\n"
            f"Actual:   {actual_rows:,}"
        )


print(
    "PASS: All seven source row counts match "
    "2015-2016 CDC expectations."
)


# =================================================================================================
# 9. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(
    dataset,
    dataset_name
):

    print(
        f"\n--- SEQN CHECK: {dataset_name} ---"
    )


    if "SEQN" not in dataset.columns:

        raise ValueError(
            f"{dataset_name}: SEQN is missing."
        )


    blank_count = int(

        dataset["SEQN"]
        .eq("")
        .sum()

    )


    duplicate_count = int(

        dataset["SEQN"]
        .duplicated()
        .sum()

    )


    dot_zero_count = int(

        (
            dataset["SEQN"].ne("")
            &
            dataset["SEQN"]
            .str.endswith(".0")
        )

        .sum()

    )


    invalid_format_count = int(

        (
            dataset["SEQN"].ne("")
            &
            ~dataset["SEQN"]
            .str.fullmatch(r"\d+")
        )

        .sum()

    )


    print(
        "Rows:",
        f"{len(dataset):,}"
    )


    print(
        "Unique SEQN:",
        f"{dataset['SEQN'].nunique():,}"
    )


    print(
        "Blank SEQN:",
        blank_count
    )


    print(
        "Duplicate SEQN:",
        duplicate_count
    )


    print(
        "SEQN ending '.0':",
        dot_zero_count
    )


    print(
        "Invalid SEQN format:",
        invalid_format_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{dataset_name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{dataset_name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{dataset_name}: SEQN still contains '.0'."
        )


    if invalid_format_count != 0:

        raise ValueError(
            f"{dataset_name}: invalid SEQN formatting."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 10. VALIDATE SEQN IN ALL SEVEN SOURCES
# =================================================================================================

for name, dataset in datasets.items():

    validate_seqn(
        dataset,
        name
    )


print(
    "\nPASS: SEQN integrity validated "
    "in all seven Lab files."
)


# =================================================================================================
# 11. VERIFY IMPORTANT 2015-2016 SOURCE STRUCTURE
# =================================================================================================

#
# GLU_I should NOT contain PHAFSTHR / PHAFSTMN.
#
# This is the design change that caused the old program to fail.
#

for col in [
    "PHAFSTHR",
    "PHAFSTMN"
]:

    if col in glucose.columns:

        raise ValueError(
            f"Unexpected {col} found in GLU_I.\n"
            "2015-2016 GLU_I should not contain "
            "fasting-duration variables."
        )


print(
    "\nPASS: GLU_I correctly does NOT contain "
    "PHAFSTHR / PHAFSTMN."
)


# -------------------------------------------------------------------------------------------------
# Required GLU_I fields
# -------------------------------------------------------------------------------------------------

for col in [

    "SEQN",

    "WTSAF2YR",

    "LBXGLU",

    "LBDGLUSI"

]:

    if col not in glucose.columns:

        raise ValueError(
            f"Expected GLU_I field missing: {col}"
        )


# -------------------------------------------------------------------------------------------------
# Required INS_I fields
# -------------------------------------------------------------------------------------------------

for col in [

    "SEQN",

    "WTSAF2YR",

    "LBXIN",

    "LBDINSI",

    "LBDINLC",

    "PHAFSTHR",

    "PHAFSTMN"

]:

    if col not in insulin.columns:

        raise ValueError(
            f"Expected INS_I field missing: {col}"
        )


# -------------------------------------------------------------------------------------------------
# Required TRIGLY_I fields
# -------------------------------------------------------------------------------------------------

for col in [

    "SEQN",

    "WTSAF2YR",

    "LBXTR",

    "LBDTRSI",

    "LBDLDL",

    "LBDLDLSI"

]:

    if col not in trigly.columns:

        raise ValueError(
            f"Expected TRIGLY_I field missing: {col}"
        )


# -------------------------------------------------------------------------------------------------
# Required FASTQX_I fields
# -------------------------------------------------------------------------------------------------

for col in [

    "SEQN",

    "PHAFSTHR",

    "PHAFSTMN"

]:

    if col not in fasting.columns:

        raise ValueError(
            f"Expected FASTQX_I field missing: {col}"
        )


# -------------------------------------------------------------------------------------------------
# Important study variable
# -------------------------------------------------------------------------------------------------

if "LBXGH" not in ghb.columns:

    raise ValueError(
        "LBXGH missing from GHB_I."
    )


print(
    "PASS: Key 2015-2016 Lab structures validated."
)


# =================================================================================================
# 12. CDC KEY SAMPLE-SIZE / NONMISSING CHECKS
# =================================================================================================

def count_nonblank(
    dataset,
    column
):

    if column not in dataset.columns:

        raise ValueError(
            f"Missing field: {column}"
        )


    return int(
        dataset[column]
        .ne("")
        .sum()
    )


# -------------------------------------------------------------------------------------------------
# GLU_I
# -------------------------------------------------------------------------------------------------

if count_nonblank(
    glucose,
    "LBXGLU"
) != 2972:

    raise ValueError(
        "GLU_I LBXGLU nonmissing count differs from CDC."
    )


if count_nonblank(
    glucose,
    "LBDGLUSI"
) != 2972:

    raise ValueError(
        "GLU_I LBDGLUSI nonmissing count differs from CDC."
    )


# -------------------------------------------------------------------------------------------------
# INS_I
# -------------------------------------------------------------------------------------------------

if count_nonblank(
    insulin,
    "LBXIN"
) != 2921:

    raise ValueError(
        "INS_I LBXIN nonmissing count differs from CDC."
    )


if count_nonblank(
    insulin,
    "LBDINSI"
) != 2921:

    raise ValueError(
        "INS_I LBDINSI nonmissing count differs from CDC."
    )


if count_nonblank(
    insulin,
    "LBDINLC"
) != 2921:

    raise ValueError(
        "INS_I LBDINLC nonmissing count differs from CDC."
    )


if count_nonblank(
    insulin,
    "PHAFSTHR"
) != 3135:

    raise ValueError(
        "INS_I PHAFSTHR nonmissing count differs from CDC."
    )


if count_nonblank(
    insulin,
    "PHAFSTMN"
) != 3135:

    raise ValueError(
        "INS_I PHAFSTMN nonmissing count differs from CDC."
    )


# -------------------------------------------------------------------------------------------------
# TRIGLY_I
# -------------------------------------------------------------------------------------------------

if count_nonblank(
    trigly,
    "LBXTR"
) != 2723:

    raise ValueError(
        "TRIGLY_I LBXTR nonmissing count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDTRSI"
) != 2723:

    raise ValueError(
        "TRIGLY_I LBDTRSI nonmissing count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDLDL"
) != 2699:

    raise ValueError(
        "TRIGLY_I LBDLDL nonmissing count differs from CDC."
    )


if count_nonblank(
    trigly,
    "LBDLDLSI"
) != 2699:

    raise ValueError(
        "TRIGLY_I LBDLDLSI nonmissing count differs from CDC."
    )


# -------------------------------------------------------------------------------------------------
# FASTQX_I
# -------------------------------------------------------------------------------------------------

if count_nonblank(
    fasting,
    "PHAFSTHR"
) != 8911:

    raise ValueError(
        "FASTQX_I PHAFSTHR nonmissing count differs from CDC."
    )


if count_nonblank(
    fasting,
    "PHAFSTMN"
) != 8911:

    raise ValueError(
        "FASTQX_I PHAFSTMN nonmissing count differs from CDC."
    )


# -------------------------------------------------------------------------------------------------
# GHB_I
# -------------------------------------------------------------------------------------------------

if count_nonblank(
    ghb,
    "LBXGH"
) != 6326:

    raise ValueError(
        "GHB_I LBXGH nonmissing count differs from CDC."
    )


# -------------------------------------------------------------------------------------------------
# TCHOL_I
# -------------------------------------------------------------------------------------------------

if "LBXTC" in chol.columns:

    if count_nonblank(
        chol,
        "LBXTC"
    ) != 7256:

        raise ValueError(
            "TCHOL_I LBXTC nonmissing count differs from CDC."
        )


if "LBDTCSI" in chol.columns:

    if count_nonblank(
        chol,
        "LBDTCSI"
    ) != 7256:

        raise ValueError(
            "TCHOL_I LBDTCSI nonmissing count differs from CDC."
        )


# -------------------------------------------------------------------------------------------------
# HDL_I
# -------------------------------------------------------------------------------------------------

if "LBDHDD" in hdl.columns:

    if count_nonblank(
        hdl,
        "LBDHDD"
    ) != 7256:

        raise ValueError(
            "HDL_I LBDHDD nonmissing count differs from CDC."
        )


if "LBDHDDSI" in hdl.columns:

    if count_nonblank(
        hdl,
        "LBDHDDSI"
    ) != 7256:

        raise ValueError(
            "HDL_I LBDHDDSI nonmissing count differs from CDC."
        )


print(
    "\nPASS: Key Lab result counts match CDC."
)


# =================================================================================================
# 13. CDC WTSAF2YR CHECK
# =================================================================================================

#
# Official 2015-2016 fasting subsample:
#
#   rows                 = 3,191
#   positive WTSAF2YR    = 2,743
#   WTSAF2YR = 0         =   448
#   missing              =     0
#

def validate_wtsaf2yr(
    dataset,
    dataset_name
):

    if "WTSAF2YR" not in dataset.columns:

        raise ValueError(
            f"{dataset_name}: WTSAF2YR missing."
        )


    values = pd.to_numeric(

        dataset["WTSAF2YR"]
        .where(
            dataset["WTSAF2YR"] != "",
            np.nan
        ),

        errors="raise"

    )


    positive_count = int(

        (
            values > 0
        )

        .sum()

    )


    zero_count = int(

        values
        .eq(0)
        .sum()

    )


    missing_count = int(

        values
        .isna()
        .sum()

    )


    print(
        f"\n--- WTSAF2YR: {dataset_name} ---"
    )


    print(
        "Positive:",
        f"{positive_count:,}"
    )


    print(
        "Zero:",
        f"{zero_count:,}"
    )


    print(
        "Missing:",
        f"{missing_count:,}"
    )


    if positive_count != 2743:

        raise ValueError(
            f"{dataset_name}: WTSAF2YR positive count "
            "differs from CDC."
        )


    if zero_count != 448:

        raise ValueError(
            f"{dataset_name}: WTSAF2YR zero count "
            "differs from CDC."
        )


    if missing_count != 0:

        raise ValueError(
            f"{dataset_name}: WTSAF2YR missing count "
            "differs from CDC."
        )


    print(
        "PASS"
    )


validate_wtsaf2yr(
    glucose,
    "GLU_I"
)


validate_wtsaf2yr(
    insulin,
    "INS_I"
)


validate_wtsaf2yr(
    trigly,
    "TRIGLY_I"
)


print(
    "\nPASS: WTSAF2YR CDC distribution validated "
    "in GLU_I, INS_I and TRIGLY_I."
)


# =================================================================================================
# 14. DISPLAY ALL CROSS-FILE VARIABLE OVERLAPS
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "CROSS-FILE VARIABLE OVERLAPS"
)


print(
    "=" * 110
)


dataset_names = list(
    datasets.keys()
)


for i in range(
    len(dataset_names)
):

    for j in range(
        i + 1,
        len(dataset_names)
    ):

        left_name = (
            dataset_names[i]
        )


        right_name = (
            dataset_names[j]
        )


        overlap = sorted(

            (
                set(
                    datasets[
                        left_name
                    ].columns
                )

                &

                set(
                    datasets[
                        right_name
                    ].columns
                )
            )

            -

            {
                "SEQN"
            }

        )


        if overlap:

            print(
                f"{left_name} <-> {right_name}:",
                overlap
            )


# =================================================================================================
# 15. NUMERIC OVERLAP COMPARISON FUNCTION
# =================================================================================================

#
# Source CSVs remain strings.
#
# Numeric copies are created ONLY for comparison.
#
# Example:
#
#   "8"
#
# and
#
#   "8.0"
#
# are considered numerically equivalent.
#

def compare_numeric_variable_by_seqn(
    left_df,
    right_df,
    column,
    left_name,
    right_name,
    require_same_seqn_set=False,
    require_common_missing_match=False
):

    print(
        "\n"
        + "-" * 105
    )


    print(
        f"NUMERIC FIDELITY CHECK: "
        f"{left_name}.{column} "
        f"<-> "
        f"{right_name}.{column}"
    )


    print(
        "-" * 105
    )


    if column not in left_df.columns:

        raise ValueError(
            f"{column} missing from {left_name}."
        )


    if column not in right_df.columns:

        raise ValueError(
            f"{column} missing from {right_name}."
        )


    left_part = (

        left_df[
            [
                "SEQN",
                column
            ]
        ]

        .rename(
            columns={
                column:
                    f"{column}_LEFT"
            }
        )

    )


    right_part = (

        right_df[
            [
                "SEQN",
                column
            ]
        ]

        .rename(
            columns={
                column:
                    f"{column}_RIGHT"
            }
        )

    )


    comparison = (

        left_part

        .merge(

            right_part,

            on="SEQN",

            how="outer",

            validate="one_to_one",

            indicator=True

        )

    )


    comparison[
        f"{column}_LEFT"
    ] = (

        comparison[
            f"{column}_LEFT"
        ]

        .fillna("")

    )


    comparison[
        f"{column}_RIGHT"
    ] = (

        comparison[
            f"{column}_RIGHT"
        ]

        .fillna("")

    )


    only_left = int(

        comparison["_merge"]
        .eq("left_only")
        .sum()

    )


    only_right = int(

        comparison["_merge"]
        .eq("right_only")
        .sum()

    )


    common = (
        comparison["_merge"]
        .eq("both")
    )


    common_count = int(
        common.sum()
    )


    print(
        "Common SEQN:",
        f"{common_count:,}"
    )


    print(
        "Only in",
        left_name,
        ":",
        f"{only_left:,}"
    )


    print(
        "Only in",
        right_name,
        ":",
        f"{only_right:,}"
    )


    if require_same_seqn_set:

        if (
            only_left != 0
            or
            only_right != 0
        ):

            raise ValueError(
                f"{left_name} and {right_name} do not "
                f"contain the same SEQN set for {column}."
            )


    common_left_raw = (

        comparison.loc[
            common,
            f"{column}_LEFT"
        ]

        .reset_index(
            drop=True
        )

    )


    common_right_raw = (

        comparison.loc[
            common,
            f"{column}_RIGHT"
        ]

        .reset_index(
            drop=True
        )

    )


    left_populated = (
        common_left_raw.ne("")
    )


    right_populated = (
        common_right_raw.ne("")
    )


    common_missing_mismatch = int(

        (
            left_populated
            !=
            right_populated
        )

        .sum()

    )


    print(
        "Missing-position mismatches "
        "within common SEQN:",
        f"{common_missing_mismatch:,}"
    )


    if (
        require_common_missing_match
        and
        common_missing_mismatch != 0
    ):

        raise ValueError(
            f"{column}: missing positions differ "
            f"between {left_name} and {right_name} "
            "for common participants."
        )


    both_populated = (
        left_populated
        &
        right_populated
    )


    overlap_count = int(
        both_populated.sum()
    )


    print(
        "Common SEQN with both populated:",
        f"{overlap_count:,}"
    )


    left_numeric = pd.to_numeric(

        common_left_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    right_numeric = pd.to_numeric(

        common_right_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        left_numeric.to_numpy(
            dtype=float
        ),

        right_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch_count = int(

        (
            ~numeric_equal
        )

        .sum()

    )


    left_text = (

        common_left_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    right_text = (

        common_right_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    text_difference_count = int(

        (
            left_text
            !=
            right_text
        )

        .sum()

    )


    print(
        "Raw text-format differences:",
        f"{text_difference_count:,}"
    )


    print(
        "TRUE numeric mismatches:",
        f"{numeric_mismatch_count:,}"
    )


    if numeric_mismatch_count != 0:

        common_rows = (

            comparison.loc[
                common
            ]

            .reset_index(
                drop=True
            )

        )


        populated_rows = (

            common_rows.loc[
                both_populated.to_numpy()
            ]

            .reset_index(
                drop=True
            )

        )


        populated_rows[
            f"{column}_LEFT_NUMERIC"
        ] = (

            left_numeric

            .reset_index(
                drop=True
            )

        )


        populated_rows[
            f"{column}_RIGHT_NUMERIC"
        ] = (

            right_numeric

            .reset_index(
                drop=True
            )

        )


        populated_rows[
            "NUMERIC_MATCH"
        ] = numeric_equal


        print(
            "\nTRUE NUMERIC MISMATCH EXAMPLES:"
        )


        display(

            populated_rows.loc[
                ~populated_rows[
                    "NUMERIC_MATCH"
                ],
                [
                    "SEQN",

                    f"{column}_LEFT",

                    f"{column}_RIGHT",

                    f"{column}_LEFT_NUMERIC",

                    f"{column}_RIGHT_NUMERIC"
                ]
            ]

            .head(20)

        )


        raise ValueError(
            f"{column} has true numeric differences "
            f"between {left_name} and {right_name}."
        )


    print(
        "PASS: Numeric values agree."
    )


    return {

        "common_seqn":
            common_count,

        "overlap":
            overlap_count,

        "common_missing_mismatches":
            common_missing_mismatch,

        "text_differences":
            text_difference_count,

        "numeric_mismatches":
            numeric_mismatch_count,

        "only_left_seqn":
            only_left,

        "only_right_seqn":
            only_right

    }


# =================================================================================================
# 16. VALIDATE WTSAF2YR - GLU_I VS INS_I
# =================================================================================================

wtsaf_ins_result = (

    compare_numeric_variable_by_seqn(

        glucose,

        insulin,

        "WTSAF2YR",

        "GLU_I",

        "INS_I",

        require_same_seqn_set=True,

        require_common_missing_match=True

    )

)


# =================================================================================================
# 17. VALIDATE WTSAF2YR - GLU_I VS TRIGLY_I
# =================================================================================================

wtsaf_trigly_result = (

    compare_numeric_variable_by_seqn(

        glucose,

        trigly,

        "WTSAF2YR",

        "GLU_I",

        "TRIGLY_I",

        require_same_seqn_set=True,

        require_common_missing_match=True

    )

)


print(
    "\nPASS: WTSAF2YR is numerically equivalent "
    "across GLU_I, INS_I and TRIGLY_I."
)


print(
    "Canonical source: GLU_I"
)


# =================================================================================================
# 18. VALIDATE INS_I PHAFSTHR AGAINST FASTQX_I
# =================================================================================================

phafsthr_result = (

    compare_numeric_variable_by_seqn(

        insulin,

        fasting,

        "PHAFSTHR",

        "INS_I",

        "FASTQX_I",

        require_same_seqn_set=False,

        require_common_missing_match=True

    )

)


# =================================================================================================
# 19. VALIDATE INS_I PHAFSTMN AGAINST FASTQX_I
# =================================================================================================

phafstmn_result = (

    compare_numeric_variable_by_seqn(

        insulin,

        fasting,

        "PHAFSTMN",

        "INS_I",

        "FASTQX_I",

        require_same_seqn_set=False,

        require_common_missing_match=True

    )

)


print(
    "\nPASS: INS_I fasting-duration values agree "
    "with FASTQX_I on common participants."
)


print(
    "Canonical PHAFSTHR / PHAFSTMN source: FASTQX_I"
)


# =================================================================================================
# 20. PREPARE INS_I FOR MERGE
# =================================================================================================

#
# Remove fields already validated and intentionally represented
# by another canonical source.
#
# Keep:
#
#   LBXIN
#   LBDINSI
#   LBDINLC
#
# plus any future/non-overlapping INS_I fields.
#

INS_REDUNDANT_COLUMNS = [

    "WTSAF2YR",

    "PHAFSTHR",

    "PHAFSTMN"

]


for col in INS_REDUNDANT_COLUMNS:

    if col not in insulin.columns:

        raise ValueError(
            f"Expected INS_I duplicate field missing: {col}"
        )


insulin_for_merge = (

    insulin.drop(
        columns=INS_REDUNDANT_COLUMNS
    )

)


print(
    "\nRemoved validated redundant INS_I fields:"
)


for col in INS_REDUNDANT_COLUMNS:

    print(
        " -",
        col
    )


# =================================================================================================
# 21. PREPARE TRIGLY_I FOR MERGE
# =================================================================================================

trigly_for_merge = (

    trigly.drop(
        columns=[
            "WTSAF2YR"
        ]
    )

)


print(
    "\nRemoved validated redundant TRIGLY_I field:"
)


print(
    " - WTSAF2YR"
)


# =================================================================================================
# 22. FASTQX_I REMAINS CANONICAL
# =================================================================================================

#
# DO NOT rename:
#
#     PHAFSTHR -> PHAFSTHR_FAST
#
#     PHAFSTMN -> PHAFSTMN_FAST
#
# in 2015-2016.
#
# FASTQX_I is the canonical source, so the final names remain:
#
#     PHAFSTHR
#     PHAFSTMN
#

fasting_for_merge = (
    fasting.copy()
)


print(
    "\nFASTQX_I canonical fasting fields:"
)


print(
    " - PHAFSTHR"
)


print(
    " - PHAFSTMN"
)


# =================================================================================================
# 23. SAFE MERGE FUNCTION
# =================================================================================================

def safe_outer_merge(
    base,
    incoming,
    incoming_name
):

    overlap = sorted(

        (
            set(
                base.columns
            )

            &

            set(
                incoming.columns
            )
        )

        -

        {
            "SEQN"
        }

    )


    if overlap:

        raise ValueError(
            f"Unexpected overlapping columns before "
            f"merging {incoming_name}:\n"
            f"{overlap}"
        )


    result = (

        base.merge(

            incoming,

            on="SEQN",

            how="outer",

            validate="one_to_one"

        )

    )


    return result


# =================================================================================================
# 24. EXPECTED FINAL SEQN UNION
# =================================================================================================

expected_seqn_union = set()


for dataset in datasets.values():

    expected_seqn_union.update(
        dataset["SEQN"]
    )


print(
    "\nExpected final unique Lab SEQN:",
    f"{len(expected_seqn_union):,}"
)


#
# FASTQX_I is the broadest source here and contains
# 9,165 participants.
#

if len(expected_seqn_union) != 9165:

    raise ValueError(
        "Unexpected source SEQN union.\n"
        f"Expected: 9,165\n"
        f"Actual:   {len(expected_seqn_union):,}"
    )


print(
    "PASS: Source SEQN union = 9,165."
)


# =================================================================================================
# 25. START WITH GLU_I
# =================================================================================================

df = (
    glucose.copy()
)


# =================================================================================================
# 26. MERGE INS_I
# =================================================================================================

df = safe_outer_merge(

    df,

    insulin_for_merge,

    "INS_I"

)


print(
    "Merged INS_I"
)


# =================================================================================================
# 27. MERGE TCHOL_I
# =================================================================================================

df = safe_outer_merge(

    df,

    chol,

    "TCHOL_I"

)


print(
    "Merged TCHOL_I"
)


# =================================================================================================
# 28. MERGE FASTQX_I
# =================================================================================================

df = safe_outer_merge(

    df,

    fasting_for_merge,

    "FASTQX_I"

)


print(
    "Merged FASTQX_I"
)


# =================================================================================================
# 29. MERGE GHB_I
# =================================================================================================

df = safe_outer_merge(

    df,

    ghb,

    "GHB_I"

)


print(
    "Merged GHB_I"
)


# =================================================================================================
# 30. MERGE HDL_I
# =================================================================================================

df = safe_outer_merge(

    df,

    hdl,

    "HDL_I"

)


print(
    "Merged HDL_I"
)


# =================================================================================================
# 31. MERGE TRIGLY_I
# =================================================================================================

df = safe_outer_merge(

    df,

    trigly_for_merge,

    "TRIGLY_I"

)


print(
    "Merged TRIGLY_I"
)


# =================================================================================================
# 32. NORMALIZE OUTER-MERGE MISSING VALUES
# =================================================================================================

#
# Corrected source CSVs use blank text.
#
# Outer merge creates NaN for participants absent from a source.
#

df = (
    df.fillna("")
)


# =================================================================================================
# 33. FINAL SEQN VALIDATION
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "FINAL SEQN CHECK"
)


print(
    "=" * 110
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Unique SEQN:",
    f"{df['SEQN'].nunique():,}"
)


if len(df) != 9165:

    raise ValueError(
        "Final Lab row count differs.\n"
        f"Expected: 9,165\n"
        f"Actual:   {len(df):,}"
    )


if (
    df["SEQN"]
    .nunique()
    != 9165
):

    raise ValueError(
        "Final unique SEQN count differs from expected."
    )


if (
    df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final Lab dataset contains duplicate SEQN."
    )


if (
    df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final Lab dataset contains blank SEQN."
    )


if (
    df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final Lab dataset contains SEQN ending '.0'."
    )


if (

    ~df["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final Lab dataset contains invalid SEQN."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 34. DUPLICATE COLUMN NAME CHECK
# =================================================================================================

duplicate_column_names = (

    df.columns[
        df.columns.duplicated()
    ]

    .tolist()

)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_column_names
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 35. VERIFY ONE CANONICAL WTSAF2YR
# =================================================================================================

final_weight_columns = [

    col

    for col in df.columns

    if col.startswith(
        "WTSAF2YR"
    )

]


print(
    "\nWTSAF2YR fields:"
)


print(
    final_weight_columns
)


if final_weight_columns != [

    "WTSAF2YR"

]:

    raise ValueError(
        "Expected exactly one canonical WTSAF2YR field."
    )


print(
    "PASS: Exactly one WTSAF2YR retained from GLU_I."
)


# =================================================================================================
# 36. VERIFY CANONICAL FASTING FIELDS
# =================================================================================================

for col in [

    "PHAFSTHR",

    "PHAFSTMN"

]:

    if col not in df.columns:

        raise ValueError(
            f"Canonical FASTQX_I field missing: {col}"
        )


#
# These old-cycle fields should NOT exist.
#

BAD_FASTING_FIELDS = [

    "PHAFSTHR_FAST",

    "PHAFSTMN_FAST",

    "PHAFSTHR_INS",

    "PHAFSTMN_INS",

    "PHAFSTHR_X",

    "PHAFSTHR_Y",

    "PHAFSTMN_X",

    "PHAFSTMN_Y"

]


bad_fasting_fields_found = [

    col

    for col in BAD_FASTING_FIELDS

    if col in df.columns

]


if bad_fasting_fields_found:

    raise ValueError(
        "Old-cycle / duplicate fasting fields remain:\n"
        + str(
            bad_fasting_fields_found
        )
    )


print(
    "PASS: One canonical PHAFSTHR / PHAFSTMN pair "
    "retained from FASTQX_I."
)


# =================================================================================================
# 37. VERIFY NO MISLEADING MERGE SUFFIXES
# =================================================================================================

BAD_SUFFIXES = [

    "_LBXGH",

    "_TRIGLY",

    "_INS",

    "_CHOL",

    "_HDL",

    "_X",

    "_Y"

]


unexpected_suffix_columns = []


for col in df.columns:

    if any(

        col.endswith(
            suffix
        )

        for suffix in BAD_SUFFIXES

    ):

        unexpected_suffix_columns.append(
            col
        )


if unexpected_suffix_columns:

    raise ValueError(
        "Unexpected merge-generated suffix columns:\n"
        + str(
            unexpected_suffix_columns
        )
    )


print(
    "PASS: No misleading merge-generated suffix columns."
)


# =================================================================================================
# 38. EXACT SOURCE-TEXT FIDELITY FUNCTION
# =================================================================================================

def validate_source_text_fidelity(
    source_df,
    final_df,
    source_name,
    retained_columns
):

    print(
        f"\n--- SOURCE FIDELITY: {source_name} ---"
    )


    if not retained_columns:

        print(
            "No retained columns to validate."
        )

        return


    source_check = (

        source_df[
            ["SEQN"] + retained_columns
        ]

        .copy()

    )


    final_check = (

        final_df[
            ["SEQN"] + retained_columns
        ]

        .copy()

    )


    final_check = (

        final_check.rename(

            columns={

                col:
                    f"{col}__FINAL"

                for col in retained_columns

            }

        )

    )


    check = (

        source_check

        .merge(

            final_check,

            on="SEQN",

            how="left",

            validate="one_to_one"

        )

        .fillna("")

    )


    failures = []


    for col in retained_columns:

        mismatch_count = int(

            (
                check[col]
                !=
                check[
                    f"{col}__FINAL"
                ]
            )

            .sum()

        )


        if mismatch_count > 0:

            failures.append(
                (
                    col,
                    mismatch_count
                )
            )


    if failures:

        raise ValueError(
            f"{source_name} source-text fidelity failures:\n"
            + str(
                failures
            )
        )


    print(
        "PASS:",
        len(retained_columns),
        "retained source fields match text-for-text."
    )


# =================================================================================================
# 39. GLU_I SOURCE FIDELITY
# =================================================================================================

GLU_RETAINED = [

    col

    for col in glucose.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    glucose,

    df,

    "GLU_I",

    GLU_RETAINED

)


# =================================================================================================
# 40. INS_I SOURCE FIDELITY
# =================================================================================================

#
# Exclude intentionally deduplicated:
#
#   WTSAF2YR
#   PHAFSTHR
#   PHAFSTMN
#

INS_RETAINED = [

    col

    for col in insulin.columns

    if col not in {

        "SEQN",

        "WTSAF2YR",

        "PHAFSTHR",

        "PHAFSTMN"

    }

]


validate_source_text_fidelity(

    insulin,

    df,

    "INS_I",

    INS_RETAINED

)


# =================================================================================================
# 41. TCHOL_I SOURCE FIDELITY
# =================================================================================================

TCHOL_RETAINED = [

    col

    for col in chol.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    chol,

    df,

    "TCHOL_I",

    TCHOL_RETAINED

)


# =================================================================================================
# 42. FASTQX_I SOURCE FIDELITY
# =================================================================================================

FASTQX_RETAINED = [

    col

    for col in fasting.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    fasting,

    df,

    "FASTQX_I",

    FASTQX_RETAINED

)


# =================================================================================================
# 43. GHB_I SOURCE FIDELITY
# =================================================================================================

GHB_RETAINED = [

    col

    for col in ghb.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    ghb,

    df,

    "GHB_I",

    GHB_RETAINED

)


# =================================================================================================
# 44. HDL_I SOURCE FIDELITY
# =================================================================================================

HDL_RETAINED = [

    col

    for col in hdl.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    hdl,

    df,

    "HDL_I",

    HDL_RETAINED

)


# =================================================================================================
# 45. TRIGLY_I SOURCE FIDELITY
# =================================================================================================

#
# WTSAF2YR intentionally removed after validation.
#

TRIGLY_RETAINED = [

    col

    for col in trigly.columns

    if col not in {

        "SEQN",

        "WTSAF2YR"

    }

]


validate_source_text_fidelity(

    trigly,

    df,

    "TRIGLY_I",

    TRIGLY_RETAINED

)


# =================================================================================================
# 46. FINAL CANONICAL WTSAF2YR CDC CHECK
# =================================================================================================

final_wtsaf = pd.to_numeric(

    df["WTSAF2YR"]
    .where(
        df["WTSAF2YR"] != "",
        np.nan
    ),

    errors="raise"

)


final_wtsaf_positive = int(

    (
        final_wtsaf > 0
    )

    .sum()

)


final_wtsaf_zero = int(

    final_wtsaf
    .eq(0)
    .sum()

)


#
# Unified Lab has additional participants outside the
# fasting subsample, so blanks are expected after merge.
#

final_wtsaf_nonblank = int(

    df["WTSAF2YR"]
    .ne("")
    .sum()

)


print(
    "\n--- FINAL WTSAF2YR CHECK ---"
)


print(
    "Nonblank:",
    f"{final_wtsaf_nonblank:,}"
)


print(
    "Positive:",
    f"{final_wtsaf_positive:,}"
)


print(
    "Zero:",
    f"{final_wtsaf_zero:,}"
)


if final_wtsaf_nonblank != 3191:

    raise ValueError(
        "Final WTSAF2YR populated count differs."
    )


if final_wtsaf_positive != 2743:

    raise ValueError(
        "Final WTSAF2YR positive count differs."
    )


if final_wtsaf_zero != 448:

    raise ValueError(
        "Final WTSAF2YR zero count differs."
    )


print(
    "PASS: Final canonical WTSAF2YR matches CDC."
)


# =================================================================================================
# 47. FINAL PHAFSTHR / PHAFSTMN CHECK
# =================================================================================================

final_phafsthr_count = int(

    df["PHAFSTHR"]
    .ne("")
    .sum()

)


final_phafstmn_count = int(

    df["PHAFSTMN"]
    .ne("")
    .sum()

)


print(
    "\n--- FINAL FASTING TIME CHECK ---"
)


print(
    "PHAFSTHR populated:",
    f"{final_phafsthr_count:,}"
)


print(
    "PHAFSTMN populated:",
    f"{final_phafstmn_count:,}"
)


if final_phafsthr_count != 8911:

    raise ValueError(
        "Final PHAFSTHR count differs from FASTQX_I."
    )


if final_phafstmn_count != 8911:

    raise ValueError(
        "Final PHAFSTMN count differs from FASTQX_I."
    )


print(
    "PASS: FASTQX_I fasting-time coverage preserved."
)


# =================================================================================================
# 48. FINAL INSULIN CHECK
# =================================================================================================

if int(
    df["LBXIN"]
    .ne("")
    .sum()
) != 2921:

    raise ValueError(
        "Final LBXIN count differs from INS_I."
    )


if int(
    df["LBDINSI"]
    .ne("")
    .sum()
) != 2921:

    raise ValueError(
        "Final LBDINSI count differs from INS_I."
    )


print(
    "PASS: Insulin variables retained."
)


# =================================================================================================
# 49. FINAL GLYCOHEMOGLOBIN CHECK
# =================================================================================================

if int(
    df["LBXGH"]
    .ne("")
    .sum()
) != 6326:

    raise ValueError(
        "Final LBXGH count differs from GHB_I."
    )


print(
    "PASS: LBXGH retained with 6,326 results."
)


# =================================================================================================
# 50. REPRESENTATIVE DECIMAL-PRESERVATION CHECK
# =================================================================================================

#
# Diagnostic only.
#
# No conversion or rounding occurs.
#

REPRESENTATIVE_DECIMAL_FIELDS = [

    "WTSAF2YR",

    "LBDGLUSI",

    "LBXIN",

    "LBDINSI",

    "LBXGH",

    "LBDHDDSI",

    "LBDTRSI",

    "LBDLDLSI",

    "LBDTCSI"

]


print(
    "\n"
    + "=" * 110
)


print(
    "LAB DECIMAL PRESERVATION"
)


print(
    "=" * 110
)


for col in REPRESENTATIVE_DECIMAL_FIELDS:

    if col not in df.columns:

        continue


    populated_count = int(

        df[col]
        .ne("")
        .sum()

    )


    decimal_token_count = int(

        (
            df[col].ne("")
            &
            df[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


print(
    "PASS: Lab values remained source text; "
    "no rounding/integer conversion during merge."
)


# =================================================================================================
# 51. KNOWN XPORT TINY-ARTIFACT CHECK
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


tiny_artifact_columns = {}


for col in df.columns:

    if col == "SEQN":

        continue


    count = int(

        df[col]

        .str.lower()

        .eq(
            TINY_TOKEN.lower()
        )

        .sum()

    )


    if count > 0:

        tiny_artifact_columns[
            col
        ] = count


tiny_artifact_count = sum(
    tiny_artifact_columns.values()
)


print(
    "\nKnown tiny XPORT artifacts:",
    f"{tiny_artifact_count:,}"
)


if tiny_artifact_count != 0:

    raise ValueError(
        "Known XPORT tiny-zero artifacts remain:\n"
        + str(
            tiny_artifact_columns
        )
    )


print(
    "PASS: No known XPORT tiny-zero artifacts."
)


# =================================================================================================
# 52. SAVE FINAL LAB CSV
# =================================================================================================

#
# DO NOT USE:
#
#     float_format=
#
# DO NOT round values.
#
# All retained corrected source text is written AS IS.
#

df.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved final Lab CSV:"
)


print(
    OUTPUT_PATH
)


# =================================================================================================
# 53. RE-READ FINAL CSV AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 54. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != df.shape:

    raise ValueError(
        "CSV shape changed after save.\n"
        f"Before: {df.shape}\n"
        f"After:  {final_df.shape}"
    )


if (
    final_df.columns.tolist()
    !=
    df.columns.tolist()
):

    raise ValueError(
        "CSV column structure changed after save."
    )


print(
    "PASS: CSV structure preserved."
)


# =================================================================================================
# 55. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

post_save_failures = []


for col in df.columns:

    mismatch_count = int(

        (
            df[col]
            !=
            final_df[col]
        )

        .sum()

    )


    if mismatch_count > 0:

        post_save_failures.append(
            (
                col,
                mismatch_count
            )
        )


if post_save_failures:

    raise ValueError(
        "Text changed during CSV export:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 56. FINAL SAVED SEQN CHECK
# =================================================================================================

if (
    final_df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Saved CSV contains duplicate SEQN."
    )


if (
    final_df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Saved CSV contains blank SEQN."
    )


if (
    final_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Saved CSV contains SEQN '.0'."
    )


if (

    ~final_df["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Saved CSV contains invalid SEQN formatting."
    )


print(
    "PASS: Saved CSV SEQN integrity."
)


# =================================================================================================
# 57. FINAL DUPLICATE / OLD-CYCLE FIELD CHECK
# =================================================================================================

BAD_COLUMNS = [

    "WTSAF2YR_INS",

    "WTSAF2YR_TRIGLY",

    "WTSAF2YR_LBXGH",

    "WTSAF2YR_X",

    "WTSAF2YR_Y",

    "PHAFSTHR_FAST",

    "PHAFSTMN_FAST",

    "PHAFSTHR_INS",

    "PHAFSTMN_INS",

    "PHAFSTHR_X",

    "PHAFSTHR_Y",

    "PHAFSTMN_X",

    "PHAFSTMN_Y"

]


bad_columns_found = [

    col

    for col in BAD_COLUMNS

    if col in final_df.columns

]


if bad_columns_found:

    raise ValueError(
        "Unexpected duplicate/old-cycle fields remain:\n"
        + str(
            bad_columns_found
        )
    )


print(
    "PASS: No old-cycle or engineered duplicate fields."
)


# =================================================================================================
# 58. FINAL FIELD CHECKS
# =================================================================================================

for col in [

    "WTSAF2YR",

    "LBXGLU",

    "LBDGLUSI",

    "LBXIN",

    "LBDINSI",

    "LBDINLC",

    "LBXGH",

    "PHAFSTHR",

    "PHAFSTMN",

    "LBXTR",

    "LBDTRSI",

    "LBDLDL",

    "LBDLDLSI"

]:

    if col not in final_df.columns:

        raise ValueError(
            f"Expected final Lab variable missing: {col}"
        )


print(
    "PASS: All key 2015-2016 Lab variables present."
)


# =================================================================================================
# 59. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2015-2016 LAB MERGE COMPLETE"
)


print(
    "=" * 110
)


print(
    "Rows:",
    f"{len(final_df):,}"
)


print(
    "Columns:",
    f"{len(final_df.columns):,}"
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    ""
)


print(
    "Source files:"
)


print(
    "  GLU_I      = 3,191"
)


print(
    "  INS_I      = 3,191"
)


print(
    "  TRIGLY_I   = 3,191"
)


print(
    "  TCHOL_I    = 8,021"
)


print(
    "  HDL_I      = 8,021"
)


print(
    "  GHB_I      = 6,744"
)


print(
    "  FASTQX_I   = 9,165"
)


print(
    ""
)


print(
    "WTSAF2YR:"
)


print(
    "  Canonical source = GLU_I"
)


print(
    "  INS_I copy validated then removed"
)


print(
    "  TRIGLY_I copy validated then removed"
)


print(
    "  Nonblank = 3,191"
)


print(
    "  Positive = 2,743"
)


print(
    "  Zero     = 448"
)


print(
    ""
)


print(
    "PHAFSTHR / PHAFSTMN:"
)


print(
    "  NOT sourced from GLU_I"
)


print(
    "  INS_I overlap validated"
)


print(
    "  Canonical source = FASTQX_I"
)


print(
    "  PHAFSTHR populated = 8,911"
)


print(
    "  PHAFSTMN populated = 8,911"
)


print(
    ""
)


print(
    "Insulin:"
)


print(
    "  LBXIN populated   = 2,921"
)


print(
    "  LBDINSI populated = 2,921"
)


print(
    ""
)


print(
    "HbA1c:"
)


print(
    "  LBXGH populated = 6,326"
)


print(
    ""
)


print(
    "PHAFSTHR INS_I vs FASTQX_I numeric mismatches:",
    phafsthr_result[
        "numeric_mismatches"
    ]
)


print(
    "PHAFSTMN INS_I vs FASTQX_I numeric mismatches:",
    phafstmn_result[
        "numeric_mismatches"
    ]
)


print(
    ""
)


print(
    "Duplicate SEQN: 0"
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate column names: 0"
)


print(
    "Duplicate WTSAF2YR fields: 0"
)


print(
    "Old PHAFSTHR_FAST / PHAFSTMN_FAST fields: 0"
)


print(
    "Known XPORT tiny artifacts: 0"
)


print(
    "Lab decimal reformatting during merge: 0"
)


print(
    "Source-specific retained-field fidelity: PASS"
)


print(
    "Pre-save/post-save exact text fidelity: PASS"
)


print(
    "=" * 110
)

PASS: All seven 2015-2016 Lab source files exist.

NHANES 2015-2016 LAB MERGE


,file,rows,columns,column_names
0,GLU_I,3191,4,"[SEQN, WTSAF2YR, LBXGLU, LBDGLUSI]"
1,INS_I,3191,7,"[SEQN, WTSAF2YR, LBXIN, LBDINSI, LBDINLC, PHAF..."
2,TCHOL_I,8021,3,"[SEQN, LBXTC, LBDTCSI]"
3,FASTQX_I,9165,19,"[SEQN, PHQ020, PHACOFHR, PHACOFMN, PHQ030, PHA..."
4,GHB_I,6744,2,"[SEQN, LBXGH]"
5,HDL_I,8021,3,"[SEQN, LBDHDD, LBDHDDSI]"
6,TRIGLY_I,3191,6,"[SEQN, WTSAF2YR, LBXTR, LBDTRSI, LBDLDL, LBDLD..."



--- OFFICIAL SOURCE ROW COUNTS ---
GLU_I: expected=3,191, actual=3,191
INS_I: expected=3,191, actual=3,191
TRIGLY_I: expected=3,191, actual=3,191
TCHOL_I: expected=8,021, actual=8,021
HDL_I: expected=8,021, actual=8,021
GHB_I: expected=6,744, actual=6,744
FASTQX_I: expected=9,165, actual=9,165
PASS: All seven source row counts match 2015-2016 CDC expectations.

--- SEQN CHECK: GLU_I ---
Rows: 3,191
Unique SEQN: 3,191
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN format: 0
PASS

--- SEQN CHECK: INS_I ---
Rows: 3,191
Unique SEQN: 3,191
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN format: 0
PASS

--- SEQN CHECK: TCHOL_I ---
Rows: 8,021
Unique SEQN: 8,021
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN format: 0
PASS

--- SEQN CHECK: FASTQX_I ---
Rows: 9,165
Unique SEQN: 9,165
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN format: 0
PASS

--- SEQN CHECK: GHB_I ---
Rows: 6,744
Unique SEQN: 6,744
Blank SEQN: 0
Duplica